## Introducción a LangGraph

LangGraph es una biblioteca que permite construir aplicaciones complejas y con estado utilizando modelos de lenguaje (LLMs) mediante la creación de grafos dirigidos. Facilita la orquestación de múltiples pasos y la gestión del estado entre ellos, lo que es ideal para agentes y flujos de trabajo conversacionales. Es una extensión de LangChain.

In [6]:
import operator
from typing import Annotated, Sequence, TypedDict

from langchain_core.messages import BaseMessage, AIMessage # Importamos AIMessage
from langgraph.graph import StateGraph, START, END

### Definición del estado del grafo

Primero, definimos el estado que pasará entre los nodos de nuestro grafo. En este ejemplo, el estado será una lista de mensajes.

In [7]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

### Definición de los nodos

Cada nodo en el grafo es una función que toma el estado actual y devuelve una actualización del estado. Para este ejemplo, tendremos un nodo simple que simula una 'herramienta' o un 'modelo' respondiendo.

In [14]:
from langchain_core.messages import AIMessage # Importación local para mayor robustez

def agent_node(state):
    """Simula una lógica de agente o una llamada a un LLM."""
    print("---AGENT---")
    messages = state['messages']
    # Aquí iría la lógica real del LLM o de la herramienta
    # Por simplicidad, solo agregaremos un mensaje de respuesta.
    # Usamos AIMessage en lugar de BaseMessage para incluir el campo 'type' implícitamente.
    response_message = AIMessage(content="Respuesta simulada del agente al mensaje: " + messages[-1].content)
    return {"messages": [response_message]}

### Construcción del grafo

Ahora, definimos la estructura del grafo: qué nodos existen y cómo se conectan.

In [15]:
workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)

# Establecemos el punto de entrada y una transición simple
workflow.add_edge(START, "agent")
# Ahora, después de que el agente procesa, finalizamos el grafo.
workflow.add_edge("agent", END)

# Compilamos el grafo
app = workflow.compile()

### Ejecución del grafo

Finalmente, podemos invocar nuestro grafo con un mensaje inicial y ver cómo procesa el estado.

In [16]:
from langchain_core.messages import HumanMessage

# Ejecutar el grafo con un mensaje de entrada
inputs = {"messages": [HumanMessage(content="¿Hola, cómo estás?")]}

# La `app.stream` nos permite ver el estado a medida que pasa por los nodos
for s in app.stream(inputs):
    print(s)
    print("----")

print("\n--- Ejecución final ---")
final_state = app.invoke(inputs)
print(final_state['messages'][-1].content)

---AGENT---
{'agent': {'messages': [AIMessage(content='Respuesta simulada del agente al mensaje: ¿Hola, cómo estás?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}}
----

--- Ejecución final ---
---AGENT---
Respuesta simulada del agente al mensaje: ¿Hola, cómo estás?
